# Arena Trial Summary and Synchronized Video Creation

For a single block with **aligned arena events** (ms_axis timebase in `analysis/arena_events_ms/` or from `load_aligned_arena_data`):

1. **Trial summary**: For each trial (by `trial_db_id`), report whether a screen touch occurred during the trial and whether the animal was successful (reward on bug hit).
2. **Video creator**: Build synchronized trial videos (same as `trial_based_video_creation` / `synchronized_video_creation`) with **strike markers**: a vertical line on the data traces at each screen-touch time during the trial for synchronization verification.

In [7]:
from pathlib import Path
import sys
from typing import Optional, Sequence, Dict, Tuple, Union, List
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

# Ensure trial_video_with_strikes (same folder as this notebook) is importable
_vid_dir = Path.cwd() / "src" / "eye_tracking_system_tools" / "figures" / "reproduction" / "video_creation"
if not (_vid_dir / "trial_video_with_strikes.py").exists():
    _vid_dir = Path.cwd()
if (_vid_dir / "trial_video_with_strikes.py").exists() and str(_vid_dir) not in sys.path:
    sys.path.insert(0, str(_vid_dir))

from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import load_aligned_arena_data
from eye_tracking_system_tools.preprocessing.arena_alignment import (
    ARENA_EVENTS_MS_SUBDIR,
    get_arena_video_frame_offset,
)
from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df
from trial_video_with_strikes import (
    build_trial_summary,
    export_trial_video_rolling_window,
    create_synchronized_trial_videos,
)
from arena_video_export import (
    export_arena_left_10s_around_first_bug,
    export_sync_clip_arena_and_eyes,
)

%matplotlib inline

In [8]:
# Block and paths
animal = "PV_208"
date = "2025_12_14"
block_num = "019"
base_path = Path(r"D:\sample_data_for_eye_repo")
block_path = base_path / animal / date / f"block_{block_num}"
arena_videos_dir = block_path / "arena_videos"

## Minimal arena-only clip (no OE, no eye)

Uses the same sync logic as `arena_sync_verification.ipynb`: PC clock only (bug_trajectory + left camera frames_timestamps). Exports **10 s** of left arena video centered on the first bug appearance. Validate this clip first (e.g. compare with frames exported in the verification notebook) before using the full trial video pipeline below.

In [9]:
# Optional: run minimal export (requires arena_videos_dir from config above)
out_clip = export_arena_left_10s_around_first_bug(arena_videos_dir, duration_sec=10.0)
print("Written:", out_clip)

Written: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\arena_videos\videos\left_first_bug_10s.mp4


## Load block and aligned arena data

Prefer ms_axis-aligned CSVs from `analysis/arena_events_ms/` if present; otherwise use `load_aligned_arena_data(block, arena_videos_dir)`.

In [3]:
def load_aligned_arena_data_for_block(block, arena_videos_dir: Path) -> Dict[str, pd.DataFrame]:
    """Load aligned arena CSVs: from analysis/arena_events_ms/ if present, else load_aligned_arena_data."""
    out_dir = Path(block.analysis_path) / ARENA_EVENTS_MS_SUBDIR
    if out_dir.is_dir():
        arena_data = {}
        for stem, fname in [("bug_trajectory", "bug_trajectory_ms_axis.csv"),
                            ("trials_data", "trials_data_ms_axis.csv"),
                            ("screen_touches", "screen_touches_ms_axis.csv")]:
            p = out_dir / fname
            if p.exists():
                arena_data[stem] = pd.read_csv(p)
        if arena_data:
            print(f"Loaded aligned arena data from {out_dir} ({list(arena_data.keys())})")
            return arena_data
    return load_aligned_arena_data(block, arena_videos_dir)

block = BlockSync(animal, date, block_num, base_path)
block.channeldict = {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"}
block.parse_open_ephys_events()
load_final_sync_df(block)
block.final_sync_df['ms_axis'] = block.final_sync_df['Arena_TTL'].values / (block.sample_rate / 1000)

# Eye data and videos (required for video export)
block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data_degrees_raw_verified.csv')
block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv')
block.handle_eye_videos()
block.handle_arena_files()
block.calibrate_pixel_size(10)
if 'pupil_diameter' not in block.left_eye_data.columns:
    block.left_eye_data['pupil_diameter_pixels'] = block.left_eye_data.major_ax
    block.right_eye_data['pupil_diameter_pixels'] = block.right_eye_data.major_ax
    block.left_eye_data['pupil_diameter'] = block.left_eye_data['pupil_diameter_pixels'] * block.L_pix_size
    block.right_eye_data['pupil_diameter'] = block.right_eye_data['pupil_diameter_pixels'] * block.R_pix_size

# Recenter eye angles
def recenter_eye_angles_to_rest(eye_df, phi_col="k_phi", theta_col="k_theta"):
    df = eye_df.copy()
    for col, med in [(phi_col, df[phi_col].median()), (theta_col, df[theta_col].median())]:
        df[col] = df[col] - med
        df[f"{col}_recentered"] = df[col]
    return df
block.left_eye_data_centered = recenter_eye_angles_to_rest(block.left_eye_data)
block.right_eye_data_centered = recenter_eye_angles_to_rest(block.right_eye_data)

arena_data = load_aligned_arena_data_for_block(block, arena_videos_dir)
trials_df = arena_data.get("trials_data")
screen_touches_df = arena_data.get("screen_touches")
bug_traj_df = arena_data.get("bug_trajectory")
if trials_df is None:
    raise ValueError("trials_data not found in aligned arena data")

instantiated block number 019 at Path: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019, new OE version
Found the sample rate for block 019 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\oe_files\PV208_d5t2_2025-12-14_12-29-53\Record Node 106...

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 019
got it!
running parse_open_ephys_events...
block 019 has a parsed events file, reading...
[INFO] Found multiple sync files: ['final_sync_df.csv', 'blocksync_df.csv']. Using newest: final_sync_df.csv
[OK] Loaded final_sync_df.csv → block.final_sync_df (rows=113,056)
handling eye video files
converting videos...
h264 files found: 2; already have .mp4 (skip): 2;

## Sync clip (arena + eye cameras) with log

Exports the same ~10 s around first bug with **left arena**, **left eye**, and **right eye** aligned via `final_sync_df` (ms_axis). Output goes to `block_path/analysis/sync_video_trials/` by default, with a **.log** file listing frame ranges per source and the arena timestamp shift (OE_ms = PC_ms - shift_ms; ms_from_rec_start = (Arena_TTL / sample_rate) * 1000). Requires block with `final_sync_df` and eye videos loaded.

In [ ]:
# Run after block is loaded (load_block_and_arena). Default out_dir = block_path/analysis/sync_video_trials
video_path, log_path = export_sync_clip_arena_and_eyes(block, duration_sec=10.0, out_basename="first_bug_10s")
print("Video:", video_path)
print("Log: ", log_path)

## 1. Trial summary

For each trial: touch during trial (yes/no), success (any reward on bug hit during trial), and strike times (ms relative to trial start) for video markers. Uses `build_trial_summary` from `trial_video_with_strikes`.

In [4]:
trial_summary_df = build_trial_summary(trials_df, screen_touches_df)
display(trial_summary_df)

,trial_db_id,touch_during_trial,success,n_touches,strike_times_ms_rel
0,10097,False,False,0,[]
1,10098,True,True,1,[14801.0]
2,10099,True,True,1,[27330.0]
3,10100,True,True,1,[52235.0]
4,10101,False,False,0,[]
5,10102,True,True,1,[57118.0]
6,10103,True,True,1,[53298.0]
7,10104,False,False,0,[]
8,10105,False,False,0,[]
9,10106,False,False,0,[]


## 2. Video creation (with strike markers)

Copy of the rolling-window trial video export from `synchronized_video_creation` / `trial_based_video_creation`, with **strike_times_ms_rel**: when provided, a vertical line is drawn on the trace panel at each strike time (screen touch during the trial) for synchronization verification. The high-level function `create_synchronized_trial_videos(block, trial_ids, ...)` takes the block and a list of trial IDs and exports videos for those trials, with the same control inputs as the previous version (fps, lfp_channel, trace_window_half_s, arena_video, etc.).

In [ ]:
# Video functions (MonotoneFrameReader, export_trial_video_rolling_window with strike lines,
# create_synchronized_trial_videos) are in trial_video_with_strikes — already imported above.

In [6]:
# Create synchronized videos for selected trials (magenta vertical line = strike time on traces)
trial_ids_to_export = [int(trial_summary_df["trial_db_id"].iloc[2])]  # e.g. first trial; or list of IDs
written = create_synchronized_trial_videos(
    block,
    trial_ids_to_export,
    arena_data=arena_data,
    output_dir=block.analysis_path / "trial_videos_sync_verified",
    fps=60.0,
    lfp_channel=21,
    trace_window_half_s=5.0,
    arena_video=1,  # index or name (e.g. "left")
    show_debug_prints=True,
)
print(f"Wrote {len(written)} video(s): {written}")

Exporting trial 10099 (strikes: 1)...


c:\Users\nimro\miniconda3\envs\eye_repo\lib\site-packages\pandas\core\base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
Exporting trial video (rolling):   0%|          | 0/1761 [00:00<?, ?frame/s]d:\Python_projects\PETS\src\eye_tracking_system_tools\figures\reproduction\video_creation\trial_video_with_strikes.py:320: RuntimeWarning: invalid value encountered in cast
  y_i = y_f.astype(np.int32)
Exporting trial video (rolling): 100%|██████████| 1761/1761 [07:10<00:00,  4.09frame/s]  


  Saved: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\analysis\trial_videos_sync_verified\trial_10099_sync_verified.mp4
Wrote 1 video(s): [WindowsPath('D:/sample_data_for_eye_repo/PV_208/2025_12_14/block_019/analysis/trial_videos_sync_verified/trial_10099_sync_verified.mp4')]
